# rel-f1 — GPU side

This notebook is only a runner: clone the repo, install dependencies, call the scripts in
`src/`. No logic lives here, otherwise the project would have two codebases.

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, and Internet -> **On**.
Then Run All.


## 1. Repo

In [ ]:
!rm -rf /kaggle/working/IIS-RDL-Project
%cd /kaggle/working
!git clone --depth 1 https://github.com/temelkovaelena/IIS-RDL-Project.git
%cd /kaggle/working/IIS-RDL-Project
!git log --oneline -1


## 2. Dependencies

Kaggle already ships torch. On top of it go the PyG stack and a compiled sampling backend:
`NeighborLoader` cannot sample without either `pyg-lib` or `torch-sparse`, and which wheel
fits depends on the exact torch and CUDA build, so the version is derived rather than
hard-coded.


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.is_available())


In [ ]:
!pip install -q relbench torch_geometric pytorch_frame sentence-transformers


In [ ]:
import importlib
import subprocess
import sys

import torch

TORCH = torch.__version__.split("+")[0]
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"


def sampling_backend():
    importlib.invalidate_caches()
    for name in ("pyg_lib", "torch_sparse"):
        try:
            importlib.import_module(name)
            return name
        except ImportError:
            continue
    return None


# Wheels are not published for every torch release. Try the running version first, then
# fall back to recent ones, whose builds are usually compatible.
candidates = [TORCH] + [v for v in ("2.7.0", "2.6.0", "2.5.0", "2.4.0") if v != TORCH]
for version in candidates:
    if sampling_backend():
        break
    index = f"https://data.pyg.org/whl/torch-{version}+{CUDA}.html"
    print(f"trying {index}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyg-lib", "torch-sparse", "-f", index]
    )

print("sampling backend:", sampling_backend() or "NONE")


## 3. Environment check

This has to pass before training anything. Without a sampling backend the loader fails on
the first batch, so it is better to stop here with a clear message.


In [ ]:
import relbench
import torch_frame
import torch_geometric
from relbench.modeling.graph import make_pkey_fkey_graph
from relbench.modeling.utils import get_stype_proposal

print("relbench", relbench.__version__)
print("torch_frame", torch_frame.__version__)
print("torch_geometric", torch_geometric.__version__)

backend = sampling_backend()
print("sampling backend:", backend)
assert backend is not None, (
    "No sampling backend. NeighborLoader needs pyg-lib or torch-sparse. "
    "Re-run the install cell above and check its output."
)


## 4. Build the graph

Row -> node, table -> node type, foreign key -> typed edge, timestamp -> node time. Text
columns (driver, constructor and circuit names) need an embedder supplied up front, or
torch_frame refuses to build the dataset.


In [ ]:
import sys

sys.path.insert(0, "src")
from features import text_embedder_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = relbench.load_dataset("rel-f1")
db = dataset.get_db()
col_to_stype = get_stype_proposal(db)

data, col_stats = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype,
    text_embedder_cfg=text_embedder_config(device=device),
    cache_dir="/kaggle/working/cache",
)

print(data)
print()
print("node types:", sorted(data.node_types))
print("edge types:", len(data.edge_types))


In [ ]:
# The edge type names must match the ones measured locally, otherwise the statistics in
# data/graph_stats.json describe a different graph from the one being trained on.
import json

stats = json.load(open("data/graph_stats.json"))
local = set(stats["edge_types"])
gpu = {f"{s} -{r}-> {d}" for s, r, d in data.edge_types}

print("edge types match:", local == gpu)
print("local only:", sorted(local - gpu)[:5])
print("gpu only:", sorted(gpu - local)[:5])
print("nodes:", stats["n_nodes"], "| edges:", stats["n_edges_undirected"])


## 5. One training run

One task, one seed, to see it work end to end before committing to the full sweep.

Watch for `temporal check on first batch` — it prints before any training happens, so a
leak shows up immediately rather than after nine runs.


In [ ]:
!python src/train.py --task driver-top3 --seed 0 --epochs 20 --cache-dir /kaggle/working/cache


## 6. All tasks, 3 seeds each

Nine runs. Results are appended to `output/evaluation.csv`, alongside the tabular rows that
are already in the repo.


In [ ]:
for task in ["driver-top3", "driver-dnf", "driver-position"]:
    for seed in [0, 1, 2]:
        print(f"\n===== {task}  seed {seed} =====")
        !python src/train.py --task {task} --seed {seed} --epochs 20 --cache-dir /kaggle/working/cache


## 7. Results

The comparison table, and a copy of the CSV in `/kaggle/working` so it can be downloaded
from the Output panel on the right.


In [ ]:
import pandas as pd

d = pd.read_csv("output/evaluation.csv")
print(d.pivot_table(index=["task", "metric", "split"], columns="arm", values="value").round(4))
print()
print("rows:", len(d), "| arms:", sorted(d["arm"].unique()))

d.to_csv("/kaggle/working/evaluation.csv", index=False)
print("saved to /kaggle/working/evaluation.csv")
